# Portfolio Prepare: SPA Winners (Utility) + Winners Correlation (Raw Returns)

This notebook selects SPA winners by utility (`lam=1.0`, `Base` cost) and computes correlation matrices of **raw returns** among winners.
- `1d`: universe `U17` (base + onchain)
- `4h`: universe `U12` (base only)


In [1]:
# Cell 1: Setup + Config
from __future__ import annotations

import importlib.util
import logging
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

from src.artifacts import safe_write_parquet
from src.rc_spa import build_universe_matrix, spa_better_models_by_utility

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("portfolio_prepare")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Cannot locate project root from cwd={Path.cwd()}")

RESULTS_DIR = PROJECT_ROOT / "results" / "portfolio_prepare"
CORR_DIR = RESULTS_DIR / "corr"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CORR_DIR.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "base_1d_returns": PROJECT_ROOT / "results" / "runner_exp" / "1d" / "returns_oos.parquet",
    "base_1d_bench": PROJECT_ROOT / "results" / "runner_exp" / "1d" / "bench_returns_oos.parquet",
    "onchain_1d_returns": PROJECT_ROOT / "results" / "runner_onchain" / "returns_oos.parquet",
    "base_4h_returns": PROJECT_ROOT / "results" / "runner_exp" / "4h" / "returns_oos.parquet",
    "base_4h_bench": PROJECT_ROOT / "results" / "runner_exp" / "4h" / "bench_returns_oos.parquet",
}
for name, path in INPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {name} -> {path}")

COST = "Base"
LAMBDA = 1.0
ALPHA = 0.05
PVALUE_TYPE = "consistent"
REPS = 2000
SEED = 42
STUDENTIZE = True
BLOCK_SIZE = {"1d": 10, "4h": 60}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("arch installed in current kernel:", bool(importlib.util.find_spec("arch")))


PROJECT_ROOT: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma
RESULTS_DIR: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\portfolio_prepare
arch installed in current kernel: True


In [2]:
# Cell 2: Load data + build universes

def ensure_date_col(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    if "date" not in df.columns:
        if "datetime_utc" in df.columns:
            df = df.rename(columns={"datetime_utc": "date"})
        elif "index" in df.columns:
            df = df.rename(columns={"index": "date"})
        else:
            raise KeyError(f"No date-like column found. Columns: {list(df.columns)}")
    df["date"] = pd.to_datetime(df["date"])
    return df


def assert_unique(df: pd.DataFrame, *, name: str) -> None:
    key = ["date", "symbol", "cost", "strategy_id"]
    dups = int(df.duplicated(subset=key).sum())
    if dups > 0:
        raise RuntimeError(f"{name}: duplicate rows by {key}: {dups}")


base_1d_returns = ensure_date_col(pd.read_parquet(INPUTS["base_1d_returns"]))
base_1d_bench = ensure_date_col(pd.read_parquet(INPUTS["base_1d_bench"]))
onchain_1d_returns = ensure_date_col(pd.read_parquet(INPUTS["onchain_1d_returns"]))
base_4h_returns = ensure_date_col(pd.read_parquet(INPUTS["base_4h_returns"]))
base_4h_bench = ensure_date_col(pd.read_parquet(INPUTS["base_4h_bench"]))

assert_unique(base_1d_returns, name="base_1d_returns")
assert_unique(base_1d_bench, name="base_1d_bench")
assert_unique(onchain_1d_returns, name="onchain_1d_returns")
assert_unique(base_4h_returns, name="base_4h_returns")
assert_unique(base_4h_bench, name="base_4h_bench")

returns_1d_u17 = pd.concat([base_1d_returns, onchain_1d_returns], ignore_index=True)
assert_unique(returns_1d_u17, name="returns_1d_u17")
returns_4h_u12 = base_4h_returns.copy()

UNIVERSES = {
    "1d_u17": {
        "timeframe": "1d",
        "scenario": "u17",
        "returns": returns_1d_u17,
        "bench": base_1d_bench,
    },
    "4h_u12": {
        "timeframe": "4h",
        "scenario": "u12",
        "returns": returns_4h_u12,
        "bench": base_4h_bench,
    },
}

rows = []
for key, u in UNIVERSES.items():
    df = u["returns"]
    rows.append({
        "universe": key,
        "rows": len(df),
        "symbols": df["symbol"].astype(str).nunique(),
        "strategies": df["strategy_id"].astype(str).nunique(),
        "min_date": df["date"].min(),
        "max_date": df["date"].max(),
    })

display(pd.DataFrame(rows))


,universe,rows,symbols,strategies,min_date,max_date
0,1d_u17,419574,5,17,2020-09-16 00:00:00,2026-03-15
1,4h_u12,1881792,5,12,2020-08-22 04:00:00,2026-02-22


In [3]:
# Cell 3: SPA winners by utility (better_models)

winner_rows = []

for u_key, u in UNIVERSES.items():
    tf = str(u["timeframe"])
    sc = str(u["scenario"])
    returns_long = u["returns"]
    bench_long = u["bench"]
    symbols = sorted(returns_long["symbol"].astype(str).unique().tolist())

    for sym in symbols:
        returns_alt, returns_bench = build_universe_matrix(
            returns_oos_long=returns_long,
            bench_long=bench_long,
            symbol=sym,
            cost=COST,
        )

        winners = spa_better_models_by_utility(
            returns_alt,
            returns_bench,
            lam=float(LAMBDA),
            block_size=int(BLOCK_SIZE[tf]),
            reps=int(REPS),
            seed=int(SEED),
            studentize=bool(STUDENTIZE),
            alpha=float(ALPHA),
            pvalue_type=str(PVALUE_TYPE),
        )

        winner_rows.append({
            "timeframe": tf,
            "scenario": sc,
            "universe": u_key,
            "symbol": sym,
            "cost": COST,
            "lam": float(LAMBDA),
            "T": int(len(returns_alt)),
            "universe_size": int(returns_alt.shape[1]),
            "n_winners": int(len(winners)),
            "winners": list(map(str, winners)),
        })

winners_df = pd.DataFrame(winner_rows).sort_values(["timeframe", "symbol"])
safe_write_parquet(winners_df, RESULTS_DIR / "spa_winners_lam1_base.parquet")

winners_exploded = winners_df.explode("winners", ignore_index=True)
safe_write_parquet(winners_exploded, RESULTS_DIR / "spa_winners_lam1_base_exploded.parquet")

display(winners_df)


,timeframe,scenario,universe,symbol,cost,lam,T,universe_size,n_winners,winners
0,1d,u17,1d_u17,BNBUSDT,Base,1.0,1826,12,9,"[R1:RSI_MR, R2:Bollinger_MR, R3:ZScore_MR, S1:..."
1,1d,u17,1d_u17,BTCUSDT,Base,1.0,2007,17,17,"[M1:TSMOM, R1:RSI_MR, R1OC:RSI_MR_Onchain, R2:..."
2,1d,u17,1d_u17,DOGEUSDT,Base,1.0,1280,17,14,"[R1:RSI_MR, R1OC:RSI_MR_Onchain, R2OC:Bollinge..."
3,1d,u17,1d_u17,ETHUSDT,Base,1.0,2007,17,16,"[M1:TSMOM, R1:RSI_MR, R1OC:RSI_MR_Onchain, R2:..."
4,1d,u17,1d_u17,XRPUSDT,Base,1.0,1644,17,14,"[M1:TSMOM, R1:RSI_MR, R1OC:RSI_MR_Onchain, R2:..."
5,4h,u12,4h_u12,BNBUSDT,Base,1.0,10956,12,12,"[M1:TSMOM, R1:RSI_MR, R2:Bollinger_MR, R3:ZSco..."
6,4h,u12,4h_u12,BTCUSDT,Base,1.0,12060,12,12,"[M1:TSMOM, R1:RSI_MR, R2:Bollinger_MR, R3:ZSco..."
7,4h,u12,4h_u12,DOGEUSDT,Base,1.0,7680,12,12,"[M1:TSMOM, R1:RSI_MR, R2:Bollinger_MR, R3:ZSco..."
8,4h,u12,4h_u12,ETHUSDT,Base,1.0,12060,12,12,"[M1:TSMOM, R1:RSI_MR, R2:Bollinger_MR, R3:ZSco..."
9,4h,u12,4h_u12,XRPUSDT,Base,1.0,9516,12,12,"[M1:TSMOM, R1:RSI_MR, R2:Bollinger_MR, R3:ZSco..."


In [4]:
# Cell 4: Correlation matrices of RAW returns among winners

def winner_corr_matrix(
    *,
    returns_long: pd.DataFrame,
    symbol: str,
    cost: str,
    winners: list[str],
) -> pd.DataFrame:
    if not winners:
        return pd.DataFrame()

    x = returns_long[
        (returns_long["symbol"] == symbol)
        & (returns_long["cost"] == cost)
        & (returns_long["strategy_id"].astype(str).isin(list(map(str, winners))))
    ].copy()
    if x.empty:
        return pd.DataFrame()

    wide = (
        x.pivot_table(index="date", columns="strategy_id", values="ret", aggfunc="mean")
        .sort_index()
    )

    # Keep winner order from better_models output
    cols = [w for w in winners if w in wide.columns]
    if not cols:
        return pd.DataFrame()
    wide = wide[cols]

    return wide.corr()


corr_inventory_rows = []
pair_rows = []

for r in winners_df.itertuples(index=False):
    u = UNIVERSES[f"{r.timeframe}_{r.scenario}"]
    corr = winner_corr_matrix(
        returns_long=u["returns"],
        symbol=str(r.symbol),
        cost=str(r.cost),
        winners=list(r.winners),
    )

    corr_path = CORR_DIR / f"corr_{r.timeframe}_{r.scenario}_{r.symbol}_{r.cost}_lam{LAMBDA:.1f}.parquet"
    safe_write_parquet(corr, corr_path)

    corr_inventory_rows.append({
        "timeframe": r.timeframe,
        "scenario": r.scenario,
        "symbol": r.symbol,
        "cost": r.cost,
        "lam": float(LAMBDA),
        "n_winners": int(r.n_winners),
        "corr_shape": tuple(corr.shape),
        "corr_path": str(corr_path),
    })

    if not corr.empty:
        for i, row_name in enumerate(corr.index):
            for j, col_name in enumerate(corr.columns):
                if j <= i:
                    continue
                pair_rows.append({
                    "timeframe": r.timeframe,
                    "scenario": r.scenario,
                    "symbol": r.symbol,
                    "cost": r.cost,
                    "lam": float(LAMBDA),
                    "strategy_a": str(row_name),
                    "strategy_b": str(col_name),
                    "corr": float(corr.iloc[i, j]),
                })

corr_inventory = pd.DataFrame(corr_inventory_rows)
safe_write_parquet(corr_inventory, RESULTS_DIR / "winner_corr_inventory_lam1_base.parquet")

winner_corr_pairs = pd.DataFrame(pair_rows)
safe_write_parquet(winner_corr_pairs, RESULTS_DIR / "winner_corr_pairs_lam1_base.parquet")

print("Saved correlation matrices to:", CORR_DIR)
display(corr_inventory)
display(winner_corr_pairs.head(40))


Saved correlation matrices to: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\portfolio_prepare\corr


,timeframe,scenario,symbol,cost,lam,n_winners,corr_shape,corr_path
0,1d,u17,BNBUSDT,Base,1.0,9,"(9, 9)",c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
1,1d,u17,BTCUSDT,Base,1.0,17,"(17, 17)",c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
2,1d,u17,DOGEUSDT,Base,1.0,14,"(14, 14)",c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
3,1d,u17,ETHUSDT,Base,1.0,16,"(16, 16)",c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
4,1d,u17,XRPUSDT,Base,1.0,14,"(14, 14)",c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
5,4h,u12,BNBUSDT,Base,1.0,12,"(12, 12)",c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
6,4h,u12,BTCUSDT,Base,1.0,12,"(12, 12)",c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
7,4h,u12,DOGEUSDT,Base,1.0,12,"(12, 12)",c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
8,4h,u12,ETHUSDT,Base,1.0,12,"(12, 12)",c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
9,4h,u12,XRPUSDT,Base,1.0,12,"(12, 12)",c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...


,timeframe,scenario,symbol,cost,lam,strategy_a,strategy_b,corr
0,1d,u17,BNBUSDT,Base,1.0,R1:RSI_MR,R2:Bollinger_MR,0.649324
1,1d,u17,BNBUSDT,Base,1.0,R1:RSI_MR,R3:ZScore_MR,0.758993
2,1d,u17,BNBUSDT,Base,1.0,R1:RSI_MR,S1:MAFilter_RSI_MR,0.605666
3,1d,u17,BNBUSDT,Base,1.0,R1:RSI_MR,S2:MA200Filter_Bollinger_MR,0.384157
4,1d,u17,BNBUSDT,Base,1.0,R1:RSI_MR,S3:Breakout_Confirm_MA,0.125232
5,1d,u17,BNBUSDT,Base,1.0,R1:RSI_MR,S4:MA_Confirm_TSMOM,0.306302
6,1d,u17,BNBUSDT,Base,1.0,R1:RSI_MR,S5:Ensemble_3Signals,0.393691
7,1d,u17,BNBUSDT,Base,1.0,R1:RSI_MR,T3:Donchian_Breakout,0.047361
8,1d,u17,BNBUSDT,Base,1.0,R2:Bollinger_MR,R3:ZScore_MR,0.779508
9,1d,u17,BNBUSDT,Base,1.0,R2:Bollinger_MR,S1:MAFilter_RSI_MR,0.584985


In [8]:
winner_corr_pairs.sort_values('corr')

,timeframe,scenario,symbol,cost,lam,strategy_a,strategy_b,corr
372,1d,u17,ETHUSDT,Base,1.0,S2OC:MA200Filter_Bollinger_MR_Onchain,T3:Donchian_Breakout,-0.002431
367,1d,u17,ETHUSDT,Base,1.0,S2:MA200Filter_Bollinger_MR,T3:Donchian_Breakout,-0.002431
354,1d,u17,ETHUSDT,Base,1.0,S1:MAFilter_RSI_MR,T3:Donchian_Breakout,-0.002251
361,1d,u17,ETHUSDT,Base,1.0,S1OC:MAFilter_RSI_MR_Onchain,T3:Donchian_Breakout,-0.002251
25,1d,u17,BNBUSDT,Base,1.0,S1:MAFilter_RSI_MR,T3:Donchian_Breakout,-0.002076
...,...,...,...,...,...,...,...,...
347,1d,u17,ETHUSDT,Base,1.0,S1:MAFilter_RSI_MR,S1OC:MAFilter_RSI_MR_Onchain,1.000000
362,1d,u17,ETHUSDT,Base,1.0,S2:MA200Filter_Bollinger_MR,S2OC:MA200Filter_Bollinger_MR_Onchain,1.000000
453,1d,u17,XRPUSDT,Base,1.0,S1:MAFilter_RSI_MR,S1OC:MAFilter_RSI_MR_Onchain,1.000000
464,1d,u17,XRPUSDT,Base,1.0,S2:MA200Filter_Bollinger_MR,S2OC:MA200Filter_Bollinger_MR_Onchain,1.000000


In [12]:
winner_corr_pairs['corr'].mean()

np.float64(0.3720998549463379)